In [1]:
import os 
import pandas as pd 
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(
    api_key="sk-7nLKAOapaZy49FIt6QTq0VmeBSsWF9HCAFr7MZd49qubnPU9",
    base_url="https://api.chatanywhere.tech/v1"
)

In [32]:
folder_path = "../builder/txtfile"

In [33]:
file_list = [f for f in os.listdir(folder_path)]

In [34]:
title = []
book_content = []
for f in file_list:
    title.append(f.split(".txt")[0])
    with open(os.path.join(folder_path,f),"r",encoding="utf-8") as f:
        content = f.read()
    book_content.append(content)

In [35]:
df = pd.DataFrame(
    [file_list,
    title,
    book_content]
).T

In [36]:
df.columns = ['file_name',"title","content"]

In [37]:
df 

,file_name,title,content
0,BMT65_75 - Turret Indexer Assembly - Troublesh...,BMT65_75 - Turret Indexer Assembly - Troublesh...,## BMT65/75 - Turret Indexer Assembly - Troubl...
1,BMT65_75 Turret - Live Tool Drive - Alignment.txt,BMT65_75 Turret - Live Tool Drive - Alignment,## BMT65/75 Turret - Live Tool Drive - Alignme...
2,Chip Auger - Haas Service Manual.txt,Chip Auger - Haas Service Manual,### 6 - Chip Auger\n\n\n###### https://www.haa...
3,Chip Conveyor - Lathe - Haas Service Manual.txt,Chip Conveyor - Lathe - Haas Service Manual,### 9 - Chip Conveyor - Lathe\n\n\n###### http...
4,Chip Conveyor - UMC - Haas Service Manual.txt,Chip Conveyor - UMC - Haas Service Manual,### 8 - Chip Conveyor - UMC\n\n\n###### https:...
5,Coolant Refill - Calibration.txt,Coolant Refill - Calibration,## Coolant Refill - Calibration\n\n\nhttps://w...
6,Coolant Refill - Haas Service Manual.txt,Coolant Refill - Haas Service Manual,### 10 - Coolant Refill\n\n\n###### https://ww...
7,Drive Belt - Troubleshooting Guide.txt,Drive Belt - Troubleshooting Guide,## Drive Belt - Troubleshooting Guide\n\n\nhtt...
8,Electrical Safety Door Interlocks - Troublesho...,Electrical Safety Door Interlocks - Troublesho...,## Electrical Safety Door Interlocks - Trouble...
9,Foot Pedal - Troubleshooting Guide - NGC.txt,Foot Pedal - Troubleshooting Guide - NGC,## Foot Pedal - Troubleshooting Guide - NGC\n\...


In [38]:
def analysis_mamual_book(title,content):
    response = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role":"system","content":"""我会给你一个一个零件的manual book名称和对应内容，
             #要求：
             1.请帮我提炼出来该零件的名字,
             2.以及manual book里面提供的一写solution,solution只需要是概括性的描述不是具体的操作方案，比如：`make sure the coolant tank float operates correctly`,有多个solution就提取多个，以List的形式返回,
             #Output Format:
             {
             "part_name":提取出来的零件名称,
             "solutions":["make sure the coolant tank float operates correctly",...]#提取出来的solution
             }"""
                   },
                  {"role":"user","content":f"这是manual book的title：{title},这是manual book的内容：{content}"}],
        temperature=0.01
    )
    return response.choices[0].message.content

In [39]:
result = []
for ind,row in df.iterrows():
    print(ind)
    answer = analysis_mamual_book(row['title'],row['content'])
    result.append(answer)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42


In [40]:
result 

['\n\n```json\n{\n  "part_name": "Turret Indexer Assembly",\n  "solutions": [\n    "Check the incoming air supply",\n    "Check the voltage to the turret clamp or unclamp solenoid",\n    "Check the turret clamp/unclamp proximity sensor",\n    "Check the piston, internal gears and spring assembly for damage",\n    "Check the internal gears for damage",\n    "Clear obstruction such as chips between tool turret and housing",\n    "Check if air is bypassing the turret piston quad-ring",\n    "Check the spring force",\n    "Verify the air pressure and flow during tool change",\n    "Test the clamp/unclamp proximity sensor functionality",\n    "Check servo motor cables for contamination or grounding issues",\n    "Measure turret pop-out distance during unclamp/clamp cycles",\n    "Inspect solenoid voltage during M43/M44 operations",\n    "Calculate spring clamp force using applied air pressure and piston surface area"\n  ]\n}\n```',
 '\n\n```json\n{\n  "part_name": "BMT65/75 Turret Live Tool

In [30]:
def convert_json_to_dict(content):
    if "```json" in content:
        data = content.split("```json")[1].split("```")[0]
        return eval(data)
    elif "```" in content:
        data = content.split("```")[1]
        return eval(data)
    else:
        return eval(content)
    
df['part_name'].apply(lambda x:convert_json_to_dict(x))
df['solutions'].apply(lambda x:convert_json_to_dict(x))

SyntaxError: invalid syntax (<string>, line 1)

In [28]:
df['part_name'] = [convert_json_to_dict(i)['part_name'] for i in result]
df['solutions'] = [convert_json_to_dict(i)['solutions'] for i in result] 

SyntaxError: invalid syntax (<string>, line 1)

In [29]:
for i,j in enumerate(result):
    print(i)
    convert_json_to_dict(i)['part_name']

NameError: name 'result' is not defined

In [46]:
df['part_name'] = [eval(i.split("```json")[1].split("```")[0])['part_name'] for i in result]
df['solutions'] = [eval(i.split("```json")[1].split("```")[0])['solutions'] for i in result]

In [47]:
df['part_name'].value_counts()

part_name
Turret Indexer Assembly                   3
Haas Oil Skimmer                          3
Chip Conveyor                             2
I/O PCB                                   2
Lathe Spindle                             1
Mist Condenser                            1
Proximity Sensor                          1
PSUP PCB                                  1
Servo Amplifier                           1
Sigma 5 Axis Servo Motor and Cables       1
Solenoid                                  1
Spindle Minimum Lubrication System        1
Live Tooling                              1
Spindle Non-Contact Encoder (NCE)         1
Coolant Pump Impeller                     1
Through-Tool Air Blast (TAB)              1
TSC Pump                                  1
Vector Drive                              1
VMC Side Mount Tool Changer Double Arm    1
Mechanical Bijur Lubrication Pump         1
Lathe Parts Catcher                       1
Standard Coolant System                   1
BMT65/75 Turret Live T

In [48]:
df 

,file_name,title,content,part_name,solutions
0,BMT65_75 - Turret Indexer Assembly - Troublesh...,BMT65_75 - Turret Indexer Assembly - Troublesh...,## BMT65/75 - Turret Indexer Assembly - Troubl...,Turret Indexer Assembly,"[Check the incoming air supply, Check the volt..."
1,BMT65_75 Turret - Live Tool Drive - Alignment.txt,BMT65_75 Turret - Live Tool Drive - Alignment,## BMT65/75 Turret - Live Tool Drive - Alignme...,BMT65/75 Turret Live Tool Drive,[Check the alignment of the live tool drive in...
2,Chip Auger - Haas Service Manual.txt,Chip Auger - Haas Service Manual,### 6 - Chip Auger\n\n\n###### https://www.haa...,Chip Auger,[Inspect the auger motor start capacitor for d...
3,Chip Conveyor - Lathe - Haas Service Manual.txt,Chip Conveyor - Lathe - Haas Service Manual,### 9 - Chip Conveyor - Lathe\n\n\n###### http...,Chip Conveyor,[Check for a short circuit in the motor or the...
4,Chip Conveyor - UMC - Haas Service Manual.txt,Chip Conveyor - UMC - Haas Service Manual,### 8 - Chip Conveyor - UMC\n\n\n###### https:...,Chip Conveyor,[ensure the conveyor drive cable is routed pro...
5,Coolant Refill - Calibration.txt,Coolant Refill - Calibration,## Coolant Refill - Calibration\n\n\nhttps://w...,Coolant Refill System,[Adjust the Concentrate Adjustment value based...
6,Coolant Refill - Haas Service Manual.txt,Coolant Refill - Haas Service Manual,### 10 - Coolant Refill\n\n\n###### https://ww...,Coolant Refill,[make sure the coolant tank float operates cor...
7,Drive Belt - Troubleshooting Guide.txt,Drive Belt - Troubleshooting Guide,## Drive Belt - Troubleshooting Guide\n\n\nhtt...,Drive Belt,"[Review the machine application and tooling, A..."
8,Electrical Safety Door Interlocks - Troublesho...,Electrical Safety Door Interlocks - Troublesho...,## Electrical Safety Door Interlocks - Trouble...,Electrical Safety Door Interlocks,[Replace the switch head (P/N 93-2309) if dama...
9,Foot Pedal - Troubleshooting Guide - NGC.txt,Foot Pedal - Troubleshooting Guide - NGC,## Foot Pedal - Troubleshooting Guide - NGC\n\...,Foot Pedal,[Check and clean under the foot pedal to remov...


In [6]:
## gpt_version

# df.loc[df['part_name'] == "Haas Robot Package (HRP)","part_name"] = "Haas Robot Package"
# part_category_mapping = {
#     # 主轴与传动系统（Spindle & Drive）
#     "Drive Belt": "Spindle & Drive",
#     "Lathe Spindle": "Spindle & Drive",
#     "Sigma 5 - Axis Servo Motor and Cables": "Spindle & Drive",
#     "Servo Amplifier": "Spindle & Drive",
#     "Vector Drive": "Spindle & Drive",
#     "Spindle Non-Contact Encoder (NCE)": "Spindle & Drive",
#     "BMT65/75 Turret - Live Tool Drive": "Spindle & Drive",  # 合并到 Spindle & Drive
#     "VMC Side Mount Tool Changer - Double Arm": "Spindle & Drive",  # 合并到 Spindle & Drive
# 
#     # 冷却与润滑系统（Coolant & Lubrication）
#     "Coolant Refill": "Coolant & Lubrication",
#     "High Pressure Flood Coolant": "Coolant & Lubrication",
#     "Standard Flood Coolant": "Coolant & Lubrication",
#     "Mist Condenser": "Coolant & Lubrication",
#     "Spindle Minimum Lubrication System": "Coolant & Lubrication",
#     "Mechanical Bijur Lubrication Pump": "Coolant & Lubrication",
#     "Haas Oil Skimmer": "Coolant & Lubrication",
#     "Through-Tool Air Blast (TAB)": "Coolant & Lubrication",
#     "TSC-300/1K": "Coolant & Lubrication",
# 
#     # 电气与控制系统（Electrical & Control）
#     "PSUP PCB": "Electrical & Control",
#     "I/O PCB": "Electrical & Control",
#     "Electrical Safety Door Interlocks": "Electrical & Control",
#     "Proximity Sensor": "Electrical & Control",
#     "Solenoid": "Electrical & Control",
# 
#     # 液压与气动系统（Hydraulic & Pneumatic）
#     "Hydraulic Power Unit": "Hydraulic & Pneumatic",
#     "Lathe HPU": "Hydraulic & Pneumatic",
#     "Hydraulic Tailstock": "Hydraulic & Pneumatic",
# 
#     # 机械组件（Mechanical Components）
#     "Chip Auger": "Mechanical Components",
#     "Chip Conveyor": "Mechanical Components",
#     "Way Cover": "Mechanical Components",
#     "Foot Pedal": "Mechanical Components",
#     "Lathe": "Mechanical Components",  # 合并 Lathe 到此类
# 
#     # 自动化与机器人系统（Automation & Robotics）
#     "Haas Robot Package": "Automation & Robotics",
#     "Parts Catcher": "Automation & Robotics",
#     "Turret Indexer Assembly": "Spindle & Drive",
#     "Live Tooling": "Spindle & Drive",
# }

In [22]:
response = client.chat.completions.create(
    model="deepseek-reasoner",
    messages=[{"role":"system","content":"""我将会给你一批零件的名称，你需要把以下零件归类到以下6个大类中：
    Spindle & Drive\Mechanical Components\Automation & Robotics\Coolant & Lubrication\Electrical & Control\Hydraulic & Pneumatic
    然后返回一个对应的码表,不需要有其他的解释
         #Output Format:
         {
         "零件的名称":"归类的大类名称",
         "零件的名称":"归类的大类名称",
         ...
         }"""
               },
              {"role":"user","content":f"这是零件名称：{list(set(df['part_name']))}"}],
    temperature=0.01
)

In [24]:
df['category'] = df['part_name'].map(eval(response.choices[0].message.content.strip()
                                          .split("```json")[1].split("```")[0]))

In [25]:
df['category'].value_counts()

category
Coolant & Lubrication    15
Electrical & Control      9
Mechanical Components     6
Automation & Robotics     5
Hydraulic & Pneumatic     5
Spindle & Drive           3
Name: count, dtype: int64

In [78]:
df[['part_name','category']]

,part_name,category
0,Turret Indexer Assembly,Tooling & Turret
1,BMT65/75 Turret - Live Tool Drive,Spindle & Drive
2,Chip Auger,Mechanical Components
3,Chip Conveyor,Mechanical Components
4,Chip Conveyor,Mechanical Components
5,Coolant Refill,Coolant & Lubrication
6,Coolant Refill,Coolant & Lubrication
7,Drive Belt,Spindle & Drive
8,Electrical Safety Door Interlocks,Electrical & Control
9,Foot Pedal,Mechanical Components


In [26]:
df.to_excel("../builder/classified_ds.xlsx",index=False)